## SQL Agent

In [1]:
from langgraph.graph import StateGraph, START, MessagesState, add_messages, END
from langchain.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from typing import Annotated
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, DeclarativeBase
from dotenv import load_dotenv
from sqlalchemy import text
from sqlalchemy.orm import Session
import os
from langchain.tools import tool

### DB Session

In [2]:
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL").strip()

engine = create_engine(
    DATABASE_URL,
    echo=False
)

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False,
)

class Base(DeclarativeBase):
    pass


def get_db():
    db: Session = SessionLocal()
    try:
        yield db
    finally:
        db.close()

### Tools

In [3]:
@tool(name_or_callable="Fetch_Top_100_Unique_Values_In_DB_Table_Columns")
async def fetch_unique_values_in_columns(column_names: list[str], table_name: str):
    '''
    Returns **top 1000 sorted unique values** in a database table.
    Columns in column_names must be present in the table table_name.
    This tool must not be used for columns like id, company names, phone, fax.

    Args:
    column_names: list[str]: List of column names
    table_name: str: Name of the table containing column_names

    Returns:
    dict(column_name (str): unique_values (list[str]))
    '''
    restricted_column_names = ['id', 'Employees', 'company_name', 'Phone', 'Fax', 'Website', 'Company', 'Value']
    db = next(get_db())
    result = dict()
    for column in column_names:
        if column:
            if column in restricted_column_names:
                result[column] = f'This tool returns lot of values for column {column} in table {table_name}, hence avoiding it. Please do not use this tool with this column'
            else:
                query = text(f'SELECT DISTINCT({column}) AS {column} FROM {table_name} ORDER BY {column} LIMIT 1000')
                try:
                    query_result = db.execute(query).mappings().fetchall()
                except Exception as e:
                    result[column] = f"Failed to fetch unique values of {column} from table {table_name}. Reason: {e.args[0]}"
                else:
                    result[column] = [[*r.values()][0] for r in query_result]
    return result

In [4]:
await fetch_unique_values_in_columns.ainvoke({"column_names":['Equity_Type', 'primary_industry', ""], "table_name":'investing_com.company_demographics'})

{'Equity_Type': ['ORD'],
 'primary_industry': ['Basic Materials',
  'Capital Goods',
  'Communication Services',
  'Conglomerates',
  'Consumer Cyclical',
  'Consumer Discretionary',
  'Consumer Staples',
  'Consumer/Non-Cyclical',
  'Energy',
  'Financial',
  'Healthcare',
  'Industrials',
  'Information Technology',
  'Materials',
  'Real Estate',
  'Services',
  'Technology',
  'Transportation',
  'Utilities']}

In [5]:
@tool(name_or_callable="Fetch_Top_100_Column_Values_Containing_Search_String")
async def fetch_column_values_containing_search_string(column_name: str, table_name: str, search_string: str):
    '''
    Returns sorted **Top 100** values from a DB table column containing the search_string.
    The results returned are case-insensitive.
    For Example: If search_string is ab, then values returned will be [AB, ABC, BABC, ab, abc, abcd] but not [a, ba, acb]
    Columns in column_names must be present in the table table_name.

    Args:
    column_name: str: column name to search in
    table_name: str: Name of the table containing column_name
    search_string: str: String to search for

    Returns:
    search_results (list[str])
    '''
    db = next(get_db())
    if column_name:
        query = text(f"SELECT DISTINCT({column_name}) FROM {table_name} WHERE LOWER({column_name}) REGEXP '(^|[^a-z0-9_]){search_string.lower()}([^a-z0-9_]|$)' ORDER BY {column_name}")
        try:
            query_result = db.execute(query).mappings().fetchall()[:100]
        except Exception as e:
            return f"Failed to fetch values containing {search_string} from {column_name} from table {table_name}. Reason: {e.args[0]}"
        else:
            if query_result:
                return [[*r.values()][0] for r in query_result]
            else:
                return "No matches found"
    else:
        return "colum_name argument must not be empty or None"

In [6]:
await fetch_column_values_containing_search_string.ainvoke({"column_name": "company_name", "table_name":'investing_com.company_demographics', 'search_string': 'HCL'})

['HCL Infosystems Limited', 'HCL Technologies Limited']

In [7]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
load_dotenv()

tavily_search_tool = TavilySearch(
    max_results=2,
    topic="general"
)

### Agent State

In [8]:
class AgentState(MessagesState):
    user_question: str
    is_human_input_needed: bool
    is_revision_needed: bool
    reviewer_feedback: str
    review_accuracy_score: int
    generated_sql: dict
    sql_query_generation_steps: str
    is_user_question_valid: bool
    input_guardrail_response: str
    sql_result: list[dict]
    

### System Prompt

In [9]:
DDL_JSON = {
    "tables": [
    {
    "table_name": "investing_com.company_demographics",
    "table_description": "This table contains demographic information of a company",
    "columns": [
        {
            "name": "id",
            "description": "unique url of source. It is primary key of the table",
            "type": "text"
        },
        {
            "name": "company_name",
            "description": "Legal name of the company",
            "type": "varchar(200)"
        },
        {
            "name": "Country",
            "description": "Headquarter country",
            "type": "varchar(100)"
        },
        {
            "name": "Employees",
            "description": "total full-time employee count",
            "type": "bigint"
        },
        {
            "name": "Equity_Type",
            "description": "Type of Equity",
            "type": "varchar(10)"
        },
        {
            "name": "Fax",
            "description": "company's fax number",
            "type": "varchar(100)"
        },
        {
            "name": "Phone",
            "description": "company's phone number",
            "type": "varchar(100)"
        },
        {
            "name": "primary_industry",
            "description": "company's primary (L1) industry",
            "type": "varchar(100)"
        },
        {
            "name": "secondary_industry",
            "description": "company's secondary (L2) industry",
            "type": "varchar(100)"
        },
        {
            "name": "Website",
            "description": "company's website",
            "type": "varchar(100)"
        },
        
    ]
    },
    {
    "table_name": "investing_com.company_financials",
    "table_description": "This table contains financial details of a company",
    "columns": [
        {
            "name": "id",
            "description": "unique url of source. It is primary key of the table",
            "type": "text"
        },
        {
            "name": "Currency",
            "description": "denomination and currency in which details are reported. Example: * In Millions of PLN (except for per share items)",
            "type": "text"
        },
        {
            "name": "Day",
            "description": "day of reporting (1-31)",
            "type": "integer"
        },
        {
            "name": "FY",
            "description": "Adjusted financial year. If month of reporting is before June of a year, then FY will be previous year",
            "type": "integer"
        },
         {
            "name": "Month",
            "description": "month of reporting (1-12)",
            "type": "integer"
        },
        {
            "name": "Particulars",
            "description": "name of financial entity like Revenue, Cost of Revenue, etc",
            "type": "varchar(300)"
        },
        {
            "name": "Period",
            "description": "reporting date",
            "type": "date"
        },
        {
            "name": "Value",
            "description": "amount of financial entity like Revenue, Cost of Revenue, etc",
            "type": "double"
        },
        {
            "name": "Year",
            "description": "year of reporting",
            "type": "integer"
        },
    ]
        }
        
    ]
           }

In [10]:
DDL_PROMPT = (
    "Below is the data dictionary containing the details of tables and columns in the table.\n"
    f"{DDL_JSON}\n\n"
    "ADDITIONAL INFO:\n"
    "- User question will not specify exact vocabulary as in the DB like company name, country, industry, currency, etc will not be exact as in DB, so definitely use the provided tools where needed to search data in DB before generating SQL query.\n"
    "- id column is the foreign key used to join both the tables. id column doesn't carry any business value, its just used to join tables.\n"
    "- don't return id column in any of the queries."
    "- When user asks about revenue he means `Total Revenue`.\n"
    "- Use secondary_industry for a more granular industry classification of companies.\n"
    "- Never use wildcards such as LIKE or ILIKE in the SQL query.\n"
    "- Keep the query as simple as possible.\n"
    
)

In [11]:
from pydantic import BaseModel


class SQLGenerationResponse(BaseModel):
    generated_sql: str
    sql_query_generation_steps: str


In [12]:
SYSTEM_PROMPT = (
    "You are an expert MySQL Data Analyst.\n"
    "Generate a SQL query in MySQL dialect based on the user's question.\n\n"
    "Instructions:\n"
    "- Never invent tool names.\n"
    "- Never invoke tools named json, python, or any other tool unless explicitly registered.\n"
    "- A human will review the query generated, so generate concise process/steps that you followed to generate the query. Also Mention the details of tool calls and reason for choosing a particular value like for company_name, etc\n"
    "- If any review feedback is present then follow it and make corrective changes, if necessary.\n"
    "- If the reviewer feedback is incorrect, generate the correct SQL and address the reviewer feedback in detail in the steps followed."
    f"{DDL_PROMPT}\n\n"
    "Response/Output Format:\n"
    "- Return the response as VALID JSON using the below Pydantic Model\n"
    f"{SQLGenerationResponse.model_fields}\n"
    "- Don't add any additional text before or after the JSON.\n"
    "- Don't include any unidecode characters in the response.\n"
    "- Just return the raw JSON that can be parsed using Python json.loads()\n"
)

### LLM

In [13]:
from langchain_openai import ChatOpenAI

def get_llm_instance(reasoning: bool=False):
    model = ChatOpenAI(
        model="gpt-5-mini",
        temperature=0.1,
        max_tokens=None,
        timeout=None,
        max_retries=2
    )
    return model

### Guardrail Node

In [14]:
from pydantic import BaseModel


class GuardrailResponse(BaseModel):
    is_user_question_valid: bool
    input_guardrail_response: str


In [15]:
import json

async def input_guardrail(state: AgentState):
    guardrail_llm = get_llm_instance().bind(response_format={"type": "json_object"})
    GUARDRAIL_PROMPT = (
        "You are a content filtering and guardrail expert who needs to decide whether the user query must be forwaded to a downstream agent or rejected.\n\n"
        "USER QUERY:\n"
        f"{state['user_question']}\n\n"
        "INSTRUCTIONS:\n"
        "- You need to allow questions like company contact info, company financials, country/industry finacial analysis, etc and return is_user_question_valid = True.\n"
        "- If a user greets, then your response should be a greeting in return and return is_user_question_valid = False.\n"
        "- If a user asks some general question, then your response should be 'Invalid question' and return is_user_question_valid = False.\n\n"
        "Response Format:\n"
        "- Return the response as json using the below Pydantic Model\n"
        f"{GuardrailResponse.model_fields}\n"
        "- Don't add any additional text before or after the JSON.\n"
        "- Don't include any unidecode characters in the response.\n"
        "- Just return the raw json that can be parsed using Python json.loads()\n"
    )
    response = await guardrail_llm.ainvoke(GUARDRAIL_PROMPT)
    response_json = dict()
    if response.content:
        response_json = json.loads(response.content)
    return {"is_user_question_valid": response_json.get('is_user_question_valid', False), 
            "input_guardrail_response": response_json.get('input_guardrail_response', 'An unexpected error occured. Please try again later.')}

### Generate SQL Node

In [16]:
import json

async def generate_sql(state: AgentState):
    llm = get_llm_instance()
    TOOLS = [fetch_unique_values_in_columns, 
             fetch_column_values_containing_search_string, tavily_search_tool]
    sql_generation_llm_ = llm.bind_tools(tools=TOOLS, strict=False)
    
    max_retries = 3

    for attempt in range(max_retries + 1):
        response = await sql_generation_llm_.ainvoke(state['messages'])

        if not response.content:
            response_json = {}
            break

        try:
            response_json = json.loads(response.content)
            break

        except json.JSONDecodeError:
            if attempt == max_retries:
                response_json = dict()

    generated_sql = response_json.get('generated_sql', '')
    generated_sql = generated_sql if isinstance(generated_sql, str) else generated_sql['annotation']

    sql_query_generation_steps = response_json.get('sql_query_generation_steps', '')
    sql_query_generation_steps = sql_query_generation_steps if isinstance(sql_query_generation_steps, str) else sql_query_generation_steps['annotation']
    
    return {"messages": [response], "generated_sql": generated_sql, "sql_query_generation_steps": sql_query_generation_steps}

### SQL Review Node

In [17]:
async def execute_sql_query(query: str, limit: int=5):
    db = next(get_db())
    if query:
        try:
            query_result = db.execute(text(query)).mappings().fetchall()[:limit]
        except Exception as e:
            return f"Failed to execute the query. Reason: {e.args[0]}"
        else:
            return query_result
    else:
        return "query must not be empty"

In [18]:
await execute_sql_query("SELECT\n  cf.Year,\n  cf.Value AS Revenue,\n  LAG(cf.Value) OVER (ORDER BY cf.Year) AS PrevRevenue,\n  ((cf.Value - LAG(cf.Value) OVER (ORDER BY cf.Year)) / LAG(cf.Value) OVER (ORDER BY cf.Year)) * 100 AS YoY_Growth_Percent\nFROM\n  investing_com.company_demographics cd\nJOIN\n  investing_com.company_financials cf\n  ON cd.id = cf.id\nWHERE\n  cd.company_name = 'Infosys Limited'\n  AND cf.Particulars = 'Revenue'\nORDER BY cf.Year;")

[{'Year': 2021.0, 'Revenue': 13561.0, 'PrevRevenue': None, 'YoY_Growth_Percent': None},
 {'Year': 2022.0, 'Revenue': 16311.0, 'PrevRevenue': 13561.0, 'YoY_Growth_Percent': 20.278740505862398},
 {'Year': 2023.0, 'Revenue': 18212.0, 'PrevRevenue': 16311.0, 'YoY_Growth_Percent': 11.65471154435657},
 {'Year': 2024.0, 'Revenue': 18562.0, 'PrevRevenue': 18212.0, 'YoY_Growth_Percent': 1.9218097957390732}]

In [19]:
class SQLReviewerResponse(BaseModel):
    is_revision_needed: bool
    reviewer_feedback: str
    review_accuracy_score: int

In [20]:
async def get_tool_calls_made_by_agent(state):
    tool_calls_messages = []
    for message in state['messages']:
        if (isinstance(message, AIMessage) and message.tool_calls) or isinstance(message, ToolMessage):
            tool_calls_messages.append(message)
    return tool_calls_messages

In [21]:
async def review_sql_query(state: AgentState):
    user_question = state.get("user_question", '')
    generated_sql_query = state.get("generated_sql", '')
    sql_query_steps = state.get("sql_query_generation_steps", '')
    query_result = await execute_sql_query(state.get("generated_sql", '')) 
    tool_calls_messages = await get_tool_calls_made_by_agent(state)
    sql_reviewer_llm_ = get_llm_instance().bind(response_format={"type": "json_object"})
    
    
    prompt = (
    "You are an expert and Senior MySQL Data Analyst.\n"
    "An AI agent has generated an SQL query in MySQL dialect. You need to validate it and give a detailed feedback to the agent to regenerate the SQL Query.\n\n"
    "Below is the User Question:\n"
    f"{user_question}\n\n"
    f"{DDL_PROMPT}\n\n"
    "Below is the AI agent generated SQL Query:\n"
    f"{generated_sql_query}\n\n"
    "Below are the tool calls made by the AI Agent to generate the SQL Query:\n"
    f"{tool_calls_messages}\n\n"
    "Below are the steps followed by the AI Agent to generate the SQL Query:\n"
    f"{sql_query_steps}\n\n"
    "Below is the sample result returned by the SQL Query:\n"
    f"{query_result}\n\n"
    "INSTRUCTIONS:\n"
    "- You need to validate the generation steps and validate the query and its result to see if they answer the user question.\n"
    "- DO NOT BE VERY STRICT, ITS FINE IF QUERY RETURNS THE DESIRED RESULT.\n"
    "- Don't give any feedback regarding query optimization only ensure its correctness.\n"
    "- Ignore currency related issues as it is not important.\n"
    "- Generate a detailed feedback and corrective actions to be taken by the AI Agent to generate an accurate AQL query.\n"
    "- specify whether a revision is needed for the SQL query and give an accuracy score between 0 and 100 like 90, 95, etc\n\n"
    "Response Format:\n"
    "- Return the response as a json using the below Pydantic Model\n"
    f"{SQLReviewerResponse.model_fields}\n"
    "- Don't add any additional text before or after the JSON.\n"
    "- Don't include any unidecode characters in the response.\n"
    "- Just return the raw json that can be parsed using Python json.loads()\n"
    
)
    response = await sql_reviewer_llm_.ainvoke(prompt)
    response_json = dict()
    if response.content:
        response_json = json.loads(response.content)
        return {"messages": [response], "is_revision_needed": response_json.get('is_revision_needed', ''),
                "reviewer_feedback": response_json.get('reviewer_feedback', ''), "review_accuracy_score": response_json.get('review_accuracy_score', ''), 
               "sql_result": query_result}

In [22]:
import pandas as pd
from IPython.display import display

async def final_response(state: AgentState):
    print("--------------------\nFINAL RESPONSE\n--------------------")
    if state['is_user_question_valid']:
        print("--------------------\nResult\n--------------------")
        query_result = await execute_sql_query(state.get("generated_sql", ''), 5000)
        if not isinstance(query_result, list):
            print(query_result)
        else:
            display(pd.DataFrame(query_result))
        print("--------------------\nGenerated SQL\n--------------------")
        print(state['generated_sql'])
        print("\n--------------------\nSQL Generation Steps\n--------------------")
        print(state['sql_query_generation_steps'])
        print("\n--------------------\nSQL Accuracy\n--------------------")
        print(state['review_accuracy_score'])
        print("\n--------------------\nSQL Reviewer Feedback\n--------------------")
        print(state['reviewer_feedback'])
    else:
        print(state['input_guardrail_response'])
    return {'messages': [AIMessage(content="Agent Execution Completed Successfully.")]}

In [23]:
from typing import Literal

async def guardrail_conditional_edge(state: AgentState) -> Literal["generate_sql", "final_response"]:
    if state.get("is_user_question_valid"):
        return "generate_sql"
    else:
        return "final_response"

In [24]:
from typing import Literal

async def sql_review_conditional_edge(state: AgentState) -> Literal["generate_sql", "final_response"]:
    if state.get("is_revision_needed"):
        return "generate_sql"
    else:
        return "final_response"

In [25]:
async def tool_conditional_edge(state: AgentState) -> Literal["tools", 'review_sql_query']:
    if isinstance(state['messages'][-1], AIMessage) and state['messages'][-1].tool_calls:
        return "tools"
    else:
        return "review_sql_query"

In [26]:
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import tools_condition, ToolNode

async def get_sql_generation_agent():
    sql_generation_agent = StateGraph(AgentState)
    sql_generation_agent.add_node("input_guardrail", input_guardrail)
    sql_generation_agent.add_node("generate_sql", generate_sql)
    sql_generation_agent.add_node("tools",
                               ToolNode(tools=[
                                   fetch_unique_values_in_columns, 
                                   fetch_column_values_containing_search_string, tavily_search_tool], 
                                        handle_tool_errors=True))
    sql_generation_agent.add_node("review_sql_query", review_sql_query)
    sql_generation_agent.add_node("final_response", final_response)
    
    
    sql_generation_agent.add_edge(START, 'input_guardrail')
    sql_generation_agent.add_conditional_edges("input_guardrail", guardrail_conditional_edge)
    sql_generation_agent.add_conditional_edges("generate_sql", tool_conditional_edge)
    sql_generation_agent.add_edge("tools", "generate_sql")
    
    sql_generation_agent.add_conditional_edges("review_sql_query", sql_review_conditional_edge)
    sql_generation_agent.add_edge("final_response", END)
    
    
    sql_generation_agent = sql_generation_agent.compile()
    return sql_generation_agent


In [27]:
# (await get_sql_generation_agent()).get_graph(xray=1).draw_mermaid_png(output_file_path="graph.png")

In [28]:
async def get_response_streamer(user_query, run_id):
    sql_generation_agent = await get_sql_generation_agent()
    response_streamer = sql_generation_agent.astream(
            {
                "messages": [
                    SystemMessage(content=SYSTEM_PROMPT),
                    HumanMessage(content=f"User Question:\n{user_query}")
                ],
                "user_question": user_query
            },
            stream_mode="updates",
            config={"configurable": {"thread_id": run_id}}
        )
    return response_streamer

In [29]:
from uuid import uuid4
query = '''
Give me top 5 and bottom 5 companies based on total revenue for each year between 2021 and 2024 for countries usa, india, france and china. 
Return country, company, year, total revenue, website, column specifying top5/bottom5 
'''
response_streamer = await get_response_streamer(query, uuid4().hex)

In [30]:
async for chunk in response_streamer:
    for node_name, node_output in chunk.items():
        print(f"\n--- Node: {node_name} ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                msg.pretty_print()


--- Node: input_guardrail ---

--- Node: generate_sql ---
================================== Ai Message ==================================
Tool Calls:
  Fetch_Top_100_Column_Values_Containing_Search_String (call_mfhlvURaEb74DdA70xGFOqk1)
 Call ID: call_mfhlvURaEb74DdA70xGFOqk1
  Args:
    column_name: Country
    table_name: investing_com.company_demographics
    search_string: usa
  Fetch_Top_100_Column_Values_Containing_Search_String (call_OJpzJu9nm6CuRZwintgr8fvL)
 Call ID: call_OJpzJu9nm6CuRZwintgr8fvL
  Args:
    column_name: Country
    table_name: investing_com.company_demographics
    search_string: india
  Fetch_Top_100_Column_Values_Containing_Search_String (call_JFSftTVYfrRX7tPXWyv2OLLB)
 Call ID: call_JFSftTVYfrRX7tPXWyv2OLLB
  Args:
    column_name: Country
    table_name: investing_com.company_demographics
    search_string: france
  Fetch_Top_100_Column_Values_Containing_Search_String (call_HmCncPccHIGmsTKpsUyihZBf)
 Call ID: call_HmCncPccHIGmsTKpsUyihZBf
  Args:
    co

,country,company,year,total_revenue,website,rank_group
0,China,China Petroleum & Chemical Corporation,2021.0,2739490.000,www.sinopec.com,Top5
1,China,PetroChina Company Limited,2021.0,2614349.000,www.petrochina.com.cn,Top5
2,China,China State Construction Engineering Corporati...,2021.0,1892706.310,www.cscec.com.cn,Top5
3,China,"Ping An Insurance (Group) Company of China, Ltd.",2021.0,1273016.000,www.pingan.com,Top5
4,China,China Railway Group Limited,2021.0,1073271.720,www.crec.cn,Top5
...,...,...,...,...,...,...
155,United States,EKIMAS Corporation does not have significant o...,2024.0,0.002,NaN,Bottom5
156,United States,Blueone Card Inc,2024.0,0.004,www.blueonecard.com,Bottom5
157,United States,Aeterna Zentaris Inc.,2024.0,0.004,www.zentaris.com,Bottom5
158,United States,"United States Basketball League, Inc.",2024.0,0.020,NaN,Bottom5


--------------------
Generated SQL
--------------------
WITH agg AS (
  SELECT
    d.Country AS country,
    d.company_name AS company,
    f.Year AS year,
    SUM(f.Value) AS total_revenue,
    d.Website AS website
  FROM investing_com.company_financials f
  JOIN investing_com.company_demographics d
    ON d.id = f.id
  WHERE d.Country IN ('United States', 'India', 'France', 'China')
    AND f.Particulars = 'Total Revenue'
    AND f.Year BETWEEN 2021 AND 2024
  GROUP BY d.id, d.Country, d.company_name, d.Website, f.Year
  HAVING SUM(f.Value) IS NOT NULL
),
ranked_top AS (
  SELECT
    country,
    company,
    year,
    total_revenue,
    website,
    ROW_NUMBER() OVER (PARTITION BY country, year ORDER BY total_revenue DESC) AS rn
  FROM agg
),
ranked_bottom AS (
  SELECT
    country,
    company,
    year,
    total_revenue,
    website,
    ROW_NUMBER() OVER (PARTITION BY country, year ORDER BY total_revenue ASC) AS rn
  FROM agg
)
SELECT country,
       company,
       year,
      